# Sensitivities in an intermediate-complexity model

The tutorial computed gradients through models with a few state variables. This notebook applies the same tools to JAX-GCM (JCM), a differentiable implementation of the SPEEDY atmospheric general circulation model. SPEEDY has a full three-dimensional dynamical core and simplified parameterizations of radiation, convection, clouds, and surface fluxes. It is cheap enough to simulate a week of weather in seconds on a laptop, and complex enough that the problems from Part 3 of the tutorial start to appear.

The notebook has three parts:

1. Run JCM for seven days and look at the simulated atmosphere.
2. Make the soil drier and estimate the evaporation response, first with two nonlinear runs and then with a forward-mode derivative (JVP).
3. Warm the initial atmosphere and compare two nonlinear runs, then compute the sensitivity of land evaporation to five surface-flux parameters with a reverse-mode derivative (VJP).

Tutorial chapters that this notebook builds on:

- [1.2 First climate gradient](https://leap-stc.github.io/diff-earth/part1/first-climate-gradient.html): `jvp`, `vjp`, and when to use each.
- [2.1 Sensitivity maps](https://leap-stc.github.io/diff-earth/part2/sensitivity-maps.html): many sensitivities from one adjoint pass.
- [3.1 Gradients through chaos](https://leap-stc.github.io/diff-earth/part3/gradients-through-chaos.html) and [3.2 Memory and cost](https://leap-stc.github.io/diff-earth/part3/memory-and-cost.html): why the derivatives below use a six-hour window.
- [3.5 Do you trust your gradient?](https://leap-stc.github.io/diff-earth/part3/trusting-gradients.html): what we have not yet checked.

## Setup

Before running this notebook, set up the hackathon environment and select the **Python (diff-earth-hackathon)** kernel, following [`README.md`](README.md) in this folder. The first model run compiles, which takes about a minute; after that each run takes a few seconds.

In [ ]:
from importlib.resources import files

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from jcm.model import Model as SpeedyModel
from jcm.forcing import ForcingData
from jcm.terrain import TerrainData
from jcm import load_states_from_xarray
from jcm.physics.speedy.speedy_coords import get_speedy_coords
from jcm.physics.speedy.params import Parameters
from jcm.physics.speedy.speedy_terms import speedy_physics

import warnings
warnings.filterwarnings("ignore")


## 1. Model setup and a seven-day run

We use a T31 spectral truncation (a grid of roughly 3.75 degrees) with 8 vertical levels and a 30-minute time step. Terrain (orography and the land-sea mask) and the boundary forcing (climatologies of sea surface temperature, sea ice, land surface temperature, soil moisture, snow cover, albedo, and ozone, plus greenhouse gas concentrations) come with the `jcm` package.

`model.run` is written in JAX, so every quantity below is a differentiable function of the forcing, the initial state, and the physics parameters.

In [ ]:
# Spectral T31 grid with 8 vertical levels
coords = get_speedy_coords(spectral_truncation=31, layers=8)

# Load the terrain and climatological boundary forcing shipped with jcm
data = files("jcm") / "data/bc/t30/clim"
terrain = TerrainData.from_file(data / "terrain.nc", coords=coords)
bc = ForcingData.from_file(data / "forcing.nc", coords=coords, align_mode="wrap_year")

print("Terrain variables: ", terrain.asdict().keys())
print("Boundary variables: ", bc.asdict().keys())

In [ ]:
# Initialize model
model = SpeedyModel(
    coords = coords,
    terrain = terrain, # init terrain (e.g., orography, land mask, etc)
    time_step = 30 # model timestep in minutes
)

In [ ]:
# Run model
result = model.run(
    forcing = bc, 
    total_time = 7, # total model integration time in days
    save_interval = 0.5 # save snapshot in days
)

ds = result.to_xarray()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), constrained_layout=True)

# Plot near-surface T at final timestep
ds.temperature.isel(time=-1, level=0).plot(ax=axes[0], x="lon", y="lat", cmap="viridis", cbar_kwargs={"label": "K"})
axes[0].set(title="Near-surface air temperature", xlabel="Longitude", ylabel="Latitude")

# Plot near-surface q at final timestep
ds.specific_humidity.isel(time=-1, level=0).plot(ax=axes[1], x="lon", y="lat", cmap="viridis", cbar_kwargs={"label": "g/kg"})
axes[1].set(title="Near-surface specific humidity", xlabel="Longitude", ylabel="Latitude");

## 2a. Boundary condition: drier soil

`soilw_am` is the soil moisture availability, a climatology with one field per day of the year. We halve it everywhere and for every day, run seven days from the same start, and difference evaporation against the baseline. This is a large, finite perturbation, so the difference contains the full nonlinear response.

In [ ]:
# Soil moisture availability: (day of year, longitude, latitude)
bc.soilw_am.values.shape

In [ ]:
# Halving soil moisture 
dry_bc = bc.copy(
    soilw_am=bc.soilw_am.replace(values=0.5 * bc.soilw_am.values)
)

In [ ]:
# Run
dry = model.run(
    forcing = dry_bc, 
    total_time = 7, 
    save_interval = 0.5
)

dry_ds = dry.to_xarray()

In [ ]:
# Plot evaporation response to drier soil
(dry_ds["surface_flux.evap.0"] - ds["surface_flux.evap.0"]).isel(time=-1).plot(
    x="lon", y="lat", cmap="BrBG", center=0,
    cbar_kwargs={"label": "g/m²/s (dry − baseline)"}
)
plt.title("Evaporation response to drier soil");

## 2b. Boundary condition sensitivity with forward mode (JVP)

Now we ask a local question: how does evaporation after six hours respond to a small drying of the soil?

We write evaporation as a function of one scalar, `soil_scale`, that multiplies the soil moisture field. `jax.jvp(f, (x0,), (v,))` returns `f(x0)` and the directional derivative `(df/dx) v` in a single forward pass. With `x0 = 1` and `v = -0.01`, the tangent is the linear estimate of the evaporation change for a 1% drying, at every grid point.

This is the forward mode from [Chapter 1.2](https://leap-stc.github.io/diff-earth/part1/first-climate-gradient.html). One JVP gives the response of every output to one input direction. Drying each region separately would need one JVP per region; [Chapter 2.1](https://leap-stc.github.io/diff-earth/part2/sensitivity-maps.html) shows how reverse mode gets all of them at once for a single scalar output.

The linear estimate is only useful while the response stays close to linear. The atmosphere is chaotic, so over longer windows small perturbations grow and the linearization degrades ([Chapter 3.1](https://leap-stc.github.io/diff-earth/part3/gradients-through-chaos.html)). That is one reason we use six hours here and not seven days.

In [ ]:
def evaporation(soil_scale):
    forcing = bc.copy(soilw_am=bc.soilw_am.replace(values=soil_scale * bc.soilw_am.values))
    model = SpeedyModel(coords=coords, terrain=terrain, time_step=30)
    result = model.run(forcing=forcing, total_time=0.25, save_interval=0.25) # six-hour run
    ds = result.to_xarray()
    return ds["surface_flux.evap.0"].isel(time=-1).data

In [ ]:
# Start at the original BC; calculate the response to 1% drying.
_, soil_sensitivity = jax.jvp(evaporation, (1.0,), (-0.01,))

In [ ]:
# Plot
soil_map = ds["surface_flux.evap.0"].isel(time=-1).copy(data=np.asarray(soil_sensitivity))
soil_map.plot(
    x="lon", y="lat", cmap="RdBu_r", center=0,
    cbar_kwargs={"label": "Estimated change in evaporation (g/m²/s)"},
)
plt.title("Forward sensitivity: 1% drier soil, after 6 h");

## 3a. Initial condition: warmer atmosphere

We start from the final state of the seven-day run in section 1, warm the temperature at every level by 4 K, and run seven more days. A control run starts from the unmodified state.

Over seven days the difference between the two runs contains both a systematic response to the warming and the chaotic divergence of the weather. A single pair of runs cannot separate the two; that would need an ensemble. This is the same problem that makes long-window gradients unreliable in [Chapter 3.1](https://leap-stc.github.io/diff-earth/part3/gradients-through-chaos.html).

In [ ]:
# Warming IC
ic = load_states_from_xarray(ds.isel(time=-1).sortby("level"))
warmer_ic = ic.copy(temperature=ic.temperature + 4.0)

In [ ]:
# Matched control from the same snapshot
ic_control = model.run(
    initial_state = ic, 
    forcing = bc, 
    total_time = 7, 
    save_interval = 0.5
)
control_ds = ic_control.to_xarray()

# Run perturbed model
ic_result = model.run(
    initial_state = warmer_ic, 
    forcing = bc, 
    total_time = 7, 
    save_interval = 0.5
)

ic_ds = ic_result.to_xarray()

In [ ]:
# Plot evaporation response to warmer atmosphere
(ic_ds["surface_flux.evap.0"] - control_ds["surface_flux.evap.0"]).isel(time=-1).plot(
    x="lon", y="lat", cmap="RdBu_r", center=0,
    cbar_kwargs={"label": "g/m²/s (warmer − control)"}
)
plt.title("Evaporation response to warmer temperature");

## 3b. Parameter sensitivity with reverse mode (VJP)

The objective is the mean evaporation over land after six hours,

$$J(\theta) = \overline{E}_{\mathrm{land}}(t = 6\,\mathrm{h};\ \theta),$$

where $\theta$ is the set of five surface-flux parameters below. We want $\partial J / \partial \theta_i$ for each parameter.

| Parameter | Meaning in SPEEDY |
|---|---|
| `chl` | Heat exchange coefficient over land |
| `cdl` | Drag coefficient for momentum over land |
| `vgust` | Wind speed added for sub-grid-scale gusts (m/s) |
| `fstab` | Amplitude of the stability correction (fraction) |
| `clambda` | Heat conductivity of the skin-to-root soil layer (W/m²/K) |

`jax.vjp` runs the model forward and returns a function that propagates a sensitivity backwards through it. Seeding it with $\partial J / \partial J = 1$ gives all five derivatives from one backward pass. The cost of reverse mode does not depend on the number of parameters, which is why it is the method of choice for calibration and for sensitivity maps ([Chapters 1.2](https://leap-stc.github.io/diff-earth/part1/first-climate-gradient.html) and [2.1](https://leap-stc.github.io/diff-earth/part2/sensitivity-maps.html)).

The price is memory. The backward pass needs the forward trajectory, so memory grows with the length of the run ([Chapter 3.2](https://leap-stc.github.io/diff-earth/part3/memory-and-cost.html)). Together with chaos, this is the second reason the window is six hours.

In [ ]:
parameters = Parameters.default()
defaults = parameters.surface_flux
land = np.asarray(terrain.fmask) > 0

In [ ]:
# Define J (mean land evaporation)
def J(chl, cdl, vgust, fstab, clambda):
    surface = defaults.replace(chl=chl, cdl=cdl, vgust=vgust, fstab=fstab, clambda=clambda)
    physics = speedy_physics(parameters.replace(surface_flux=surface))
    model = SpeedyModel(coords=coords, terrain=terrain, physics=physics, time_step=30)
    result = model.run(initial_state=ic, forcing=bc, total_time=0.25, save_interval=0.25)
    ds = result.to_xarray()
    return ds["surface_flux.evap.0"].isel(time=-1).data[land].mean()

In [ ]:
# Reverse sensitivity at the default parameter values
J0, reverse = jax.vjp(
    J, defaults.chl, defaults.cdl, defaults.vgust, defaults.fstab, defaults.clambda
)
parameter_sensitivity = jnp.array(reverse(1.0)) # starts from dJ/dJ = 1.0 --> dJ/d(theta)
names = ["chl", "cdl", "vgust", "fstab", "clambda"]
units = ["dimensionless unit", "dimensionless unit", "(m/s)", "dimensionless unit", "(W/m²/K)"]
for name, value, unit in zip(names, parameter_sensitivity, units):
    print(f"dJ/d{name:<7} = {value: .3g} g/m²/s per {unit}")

The raw derivatives have different units, so their magnitudes cannot be compared. A 1 m/s change in `vgust` and a change of 1 in `chl` are not comparable perturbations. A common fix is the elasticity

$$\varepsilon_i = \frac{\theta_i}{J} \frac{\partial J}{\partial \theta_i},$$

which is dimensionless: the percent change in $J$ for a 1% change in $\theta_i$.

In [ ]:
theta = jnp.array([defaults.chl, defaults.cdl, defaults.vgust, defaults.fstab, defaults.clambda])
elasticity = theta / J0 * parameter_sensitivity

fig, ax = plt.subplots(figsize=(6, 3), constrained_layout=True)
ax.barh(names, np.asarray(elasticity))
ax.axvline(0, color="k", lw=0.8)
ax.invert_yaxis()
ax.set(xlabel="Elasticity (% change in J per 1% change in parameter)",
       title="Sensitivity of mean land evaporation after 6 h");

Compare the two views. `vgust` has one of the smallest raw derivatives, but once scaled by its default value of 5 m/s it is the second most influential parameter. Over this six-hour window, land evaporation is controlled mainly by the heat exchange coefficient and the gust speed.

## What we have not checked

Everything above takes the derivatives from JAX at face value. Before using them for science, at least three things need checking:

1. **Are the derivatives correct?** Nothing in this notebook compares them with a finite difference ([Chapter 3.5](https://leap-stc.github.io/diff-earth/part3/trusting-gradients.html)). JAX computes in single precision by default, which limits how small the finite-difference step can be.
2. **Do they hold over longer windows?** The six-hour window was chosen to avoid chaos and memory problems, not because six hours is the scientifically relevant timescale ([Chapters 3.1](https://leap-stc.github.io/diff-earth/part3/gradients-through-chaos.html) and [3.2](https://leap-stc.github.io/diff-earth/part3/memory-and-cost.html)).
3. **Are they smooth?** SPEEDY's convection and condensation schemes switch on at thresholds. Derivatives with respect to parameters of those schemes may be zero or noisy even when the model output changes ([Chapter 3.3](https://leap-stc.github.io/diff-earth/part3/non-smooth-physics.html)).

Each of these is a starting point for the hackathon challenges.